## Step 0: Install Libraries

In [21]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report


## Step 1: Load Dataset

In [22]:

# Load Dataset
df = pd.read_csv("telco.csv")


In [23]:

# Remove unnecessary column
df.drop("customerID", axis=1, inplace=True)


## Step 2: Preprocessing

In [24]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")


In [25]:
# Convert target variable
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})


## Step 3: Train / Test Split

In [26]:
# Features and target
X = df.drop("Churn", axis=1)
y = df["Churn"]

## Step 4: Build Preprocessing Pipeline

In [27]:
# Numerical and categorical columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

## Step 5: Build Full Model Pipelines

In [28]:
# Numerical pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## Step 6: Hyperparameter Tuning with GridSearchCV

In [29]:
# Categorical pipeline
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [30]:
# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Step 7: Evaluate on Test Set

In [31]:
# ===============================
# Logistic Regression Pipeline
# ===============================
log_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

log_params = {
    "classifier__C": [0.01, 0.1, 1, 10]
}

log_grid = GridSearchCV(
    log_pipeline,
    log_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

log_grid.fit(X_train, y_train)

log_preds = log_grid.predict(X_test)

print("===== Logistic Regression =====")
print("Best Parameters:", log_grid.best_params_)
print("Accuracy:", accuracy_score(y_test, log_preds))
print(classification_report(y_test, log_preds))

===== Logistic Regression =====
Best Parameters: {'classifier__C': 10}
Accuracy: 0.8055358410220014
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [32]:
# ===============================
# Random Forest Pipeline
# ===============================
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

rf_params = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)
rf_preds = rf_grid.predict(X_test)

print("\n===== Random Forest =====")
print("Best Parameters:", rf_grid.best_params_)
print("Accuracy:", accuracy_score(y_test, rf_preds))
print(classification_report(y_test, rf_preds))


===== Random Forest =====
Best Parameters: {'classifier__max_depth': 10, 'classifier__n_estimators': 100}
Accuracy: 0.8026969481902059
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.53      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409



In [33]:

best_model = rf_grid.best_estimator_

joblib.dump(best_model, "churn_pipeline.pkl")

print("\nPipeline exported successfully!")


Pipeline exported successfully!


In [34]:
from google.colab import files

files.download("churn_pipeline.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
import joblib

model = joblib.load("churn_pipeline.pkl")

print(model)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))